In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score, v_measure_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.datasets import fetch_covtype, fetch_openml
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import time
import warnings

from Algorithms import KMeansClustering, MiniBatchKMeans, FuzzyCMeans, MiniBatchFuzzyCMeans

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Part 1: Real-World Dataset Loading

We use two large-scale datasets to properly evaluate the clustering algorithms, especially the mini-batch variants where the speedup over standard algorithms becomes apparent only at scale.

**Dataset 1 — Covertype** (UCI ML Repository): ~581K samples, 54 features describing forest cover types. 7 natural classes.

**Dataset 2 — Fashion-MNIST** (Zalando Research): 70K grayscale images of clothing items, 784 features (28x28 pixels). 10 classes.

In [ ]:
covtype = fetch_covtype()
X_cov = covtype.data
y_cov = covtype.target

scaler = StandardScaler()
X_cov_scaled = scaler.fit_transform(X_cov)

print(f"Covertype: {X_cov_scaled.shape[0]} samples, {X_cov_scaled.shape[1]} features")
print(f"Classes: {np.unique(y_cov)}")
print(f"Class distribution:")
for c in np.unique(y_cov):
    print(f"  Class {c}: {np.sum(y_cov == c)} ({100*np.mean(y_cov == c):.1f}%)")

In [ ]:
fmnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False, parser='auto')
X_fm = fmnist.data.astype(np.float64)
y_fm = fmnist.target.astype(int)

X_fm_scaled = X_fm / 255.0

print(f"Fashion-MNIST: {X_fm_scaled.shape[0]} samples, {X_fm_scaled.shape[1]} features")
print(f"Classes: {np.unique(y_fm)}")

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
for i, name in enumerate(class_names):
    print(f"  {i} - {name}: {np.sum(y_fm == i)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Covertype: subsample for PCA viz
idx_cov = np.random.choice(len(X_cov_scaled), 5000, replace=False)
pca_cov = PCA(n_components=2).fit_transform(X_cov_scaled[idx_cov])
scatter1 = axes[0].scatter(pca_cov[:, 0], pca_cov[:, 1], c=y_cov[idx_cov], 
                           cmap='tab10', alpha=0.4, s=5)
axes[0].set_title('Covertype (PCA, 5K subsample)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0])

# Fashion-MNIST
idx_fm = np.random.choice(len(X_fm_scaled), 5000, replace=False)
pca_fm = PCA(n_components=2).fit_transform(X_fm_scaled[idx_fm])
scatter2 = axes[1].scatter(pca_fm[:, 0], pca_fm[:, 1], c=y_fm[idx_fm], 
                           cmap='tab10', alpha=0.4, s=5)
axes[1].set_title('Fashion-MNIST (PCA, 5K subsample)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.savefig('results/pca_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

# Part 2: Parameter Selection

The teacher stressed the importance of justifying parameter choices (k, batch size, fuzziness m). We use the Elbow method and Silhouette analysis on a subsample to determine k, then test batch size sensitivity.

In [ ]:
np.random.seed(42)

subsample_size = 10000
idx_sub = np.random.choice(len(X_cov_scaled), subsample_size, replace=False)
X_sub = X_cov_scaled[idx_sub]
y_sub = y_cov[idx_sub]

k_range = range(2, 16)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeansClustering(X_sub, n_clusters=k, max_iter=100, random_state=42)
    km.fit()
    labels = km.predict(X_sub)
    inertias.append(km.get_inertia())
    silhouettes.append(silhouette_score(X_sub, labels, sample_size=5000))
    print(f"k={k}: inertia={inertias[-1]:.0f}, silhouette={silhouettes[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, 'bo-')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method — Covertype')

axes[1].plot(list(k_range), silhouettes, 'ro-')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis — Covertype')

plt.tight_layout()
plt.savefig('results/parameter_selection_k.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
batch_sizes = [64, 128, 256, 512, 1024, 2048]
batch_results = []

for bs in batch_sizes:
    start = time.time()
    mb = MiniBatchKMeans(n_clusters=7, batch_size=bs, max_iter=100, random_state=42)
    mb.fit(X_sub)
    runtime = time.time() - start
    labels = mb.predict(X_sub)
    sil = silhouette_score(X_sub, labels, sample_size=5000)
    batch_results.append({'batch_size': bs, 'runtime': runtime, 'silhouette': sil})
    print(f"batch_size={bs}: runtime={runtime:.3f}s, silhouette={sil:.4f}")

batch_df = pd.DataFrame(batch_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar([str(b) for b in batch_sizes], batch_df['runtime'], color='steelblue')
axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Runtime (s)')
axes[0].set_title('Batch Size vs Runtime')

axes[1].plot(batch_sizes, batch_df['silhouette'], 'go-', linewidth=2)
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Batch Size vs Quality')

plt.tight_layout()
plt.savefig('results/batch_size_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

**Parameter choices based on the above analysis:**

- **k=7** for Covertype (matches the 7 known cover types, elbow/silhouette confirm)
- **k=10** for Fashion-MNIST (10 clothing categories)
- **batch_size=1024** — good tradeoff between speed and convergence quality
- **m=2.0** for FCM — standard choice, well-established in the literature
- **max_iter=300** with **tol=1e-4** — sufficient for convergence on both datasets
- **10 random seeds** per experiment for statistical robustness

In [ ]:
def evaluate_algorithm(algorithm_class, X, y_true, n_clusters, seeds=range(1, 11), **kwargs):
    results = {
        'seed': [], 'runtime': [], 'silhouette': [], 'v_measure': [], 'ari': [],
        'n_iterations': []
    }
    
    sil_sample = min(10000, len(X))
    
    for seed in seeds:
        start = time.time()
        
        if algorithm_class in [MiniBatchKMeans, MiniBatchFuzzyCMeans]:
            model = algorithm_class(n_clusters=n_clusters, random_state=seed, **kwargs)
            model.fit(X)
        else:
            model = algorithm_class(X, n_clusters=n_clusters, random_state=seed, **kwargs)
            model.fit()
        
        runtime = time.time() - start
        labels = model.predict(X)
        
        results['seed'].append(seed)
        results['runtime'].append(runtime)
        results['silhouette'].append(silhouette_score(X, labels, sample_size=sil_sample))
        results['v_measure'].append(v_measure_score(y_true, labels))
        results['ari'].append(adjusted_rand_score(y_true, labels))
        results['n_iterations'].append(seed)
    
    return pd.DataFrame(results)

# Part 3: Experiments on Covertype

Running all 4 algorithms on Covertype (~581K samples). This is where the mini-batch advantage should be most visible given the dataset size.

In [ ]:
algorithms = {
    'KMeans': (KMeansClustering, {}),
    'MiniBatch KMeans': (MiniBatchKMeans, {'batch_size': 1024}),
    'Fuzzy C-Means': (FuzzyCMeans, {'m': 2.0}),
    'MiniBatch FCM': (MiniBatchFuzzyCMeans, {'batch_size': 1024, 'm': 2.0})
}

# Use a 50K subsample for standard algorithms (full dataset is too slow for KMeans/FCM)
# Mini-batch variants run on full dataset
np.random.seed(42)
idx_50k = np.random.choice(len(X_cov_scaled), 50000, replace=False)
X_cov_50k = X_cov_scaled[idx_50k]
y_cov_50k = y_cov[idx_50k]

results_cov = {}
for name, (algo_class, params) in algorithms.items():
    print(f"\nRunning {name}...")
    if 'MiniBatch' in name:
        results_cov[name] = evaluate_algorithm(
            algo_class, X_cov_scaled, y_cov, n_clusters=7, **params
        )
    else:
        results_cov[name] = evaluate_algorithm(
            algo_class, X_cov_50k, y_cov_50k, n_clusters=7, **params
        )
    print(f"  Avg runtime: {results_cov[name]['runtime'].mean():.2f}s")
    print(f"  Avg silhouette: {results_cov[name]['silhouette'].mean():.4f}")

# Part 4: Experiments on Fashion-MNIST

Running on Fashion-MNIST (70K samples, 784 features). The high dimensionality tests algorithm scalability differently than Covertype.

In [ ]:
results_fm = {}
for name, (algo_class, params) in algorithms.items():
    print(f"\nRunning {name}...")
    results_fm[name] = evaluate_algorithm(
        algo_class, X_fm_scaled, y_fm, n_clusters=10, **params
    )
    print(f"  Avg runtime: {results_fm[name]['runtime'].mean():.2f}s")
    print(f"  Avg silhouette: {results_fm[name]['silhouette'].mean():.4f}")

In [ ]:
def summarize_results(results_dict, dataset_name):
    summary_data = []
    for algo_name, df in results_dict.items():
        summary_data.append({
            'Algorithm': algo_name,
            'Avg Runtime (s)': f"{df['runtime'].mean():.4f} +/- {df['runtime'].std():.4f}",
            'Avg Silhouette': f"{df['silhouette'].mean():.4f} +/- {df['silhouette'].std():.4f}",
            'Avg V-Measure': f"{df['v_measure'].mean():.4f} +/- {df['v_measure'].std():.4f}",
            'Avg ARI': f"{df['ari'].mean():.4f} +/- {df['ari'].std():.4f}"
        })
    return pd.DataFrame(summary_data)

print("=== Covertype Results ===")
summary_cov = summarize_results(results_cov, "Covertype")
print(summary_cov.to_string(index=False))

print("\n=== Fashion-MNIST Results ===")
summary_fm = summarize_results(results_fm, "Fashion-MNIST")
print(summary_fm.to_string(index=False))

summary_cov.to_csv('results/covertype_summary.csv', index=False)
summary_fm.to_csv('results/fmnist_summary.csv', index=False)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

datasets = [
    (results_cov, "Covertype"),
    (results_fm, "Fashion-MNIST")
]

for col, (results_dict, title) in enumerate(datasets):
    # Silhouette boxplot
    data_sil = [df['silhouette'].values for df in results_dict.values()]
    bp1 = axes[0, col].boxplot(data_sil, labels=list(results_dict.keys()), patch_artist=True)
    for patch, color in zip(bp1['boxes'], sns.color_palette("husl", 4)):
        patch.set_facecolor(color)
    axes[0, col].set_title(f'{title} — Silhouette Score')
    axes[0, col].tick_params(axis='x', rotation=30)
    
    # Runtime boxplot
    data_rt = [df['runtime'].values for df in results_dict.values()]
    bp2 = axes[1, col].boxplot(data_rt, labels=list(results_dict.keys()), patch_artist=True)
    for patch, color in zip(bp2['boxes'], sns.color_palette("husl", 4)):
        patch.set_facecolor(color)
    axes[1, col].set_title(f'{title} — Runtime (s)')
    axes[1, col].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('results/boxplots_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for col, (results_dict, title) in enumerate(datasets):
    x = np.arange(len(results_dict))
    width = 0.35
    
    vmeasures = [df['v_measure'].mean() for df in results_dict.values()]
    aris = [df['ari'].mean() for df in results_dict.values()]
    
    axes[col].bar(x - width/2, vmeasures, width, label='V-Measure', color='steelblue')
    axes[col].bar(x + width/2, aris, width, label='ARI', color='coral')
    axes[col].set_xticks(x)
    axes[col].set_xticklabels(list(results_dict.keys()), rotation=30)
    axes[col].set_title(f'{title} — Clustering Quality')
    axes[col].legend()
    axes[col].set_ylabel('Score')

plt.tight_layout()
plt.savefig('results/vmeasure_ari_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for col, (results_dict, title) in enumerate(datasets):
    for algo_name, df in results_dict.items():
        avg_runtime = df['runtime'].mean()
        avg_sil = df['silhouette'].mean()
        axes[col].scatter(avg_runtime, avg_sil, s=200, label=algo_name, zorder=5)
        axes[col].annotate(algo_name, (avg_runtime, avg_sil), 
                          textcoords="offset points", xytext=(10, 5), fontsize=9)
    
    axes[col].set_xlabel('Avg Runtime (s)')
    axes[col].set_ylabel('Avg Silhouette Score')
    axes[col].set_title(f'{title} — Runtime vs Quality')
    axes[col].legend(loc='best', fontsize=8)

plt.tight_layout()
plt.savefig('results/runtime_vs_quality.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from scipy import stats

def perform_statistical_tests(results_dict, metric='silhouette'):
    algo_names = list(results_dict.keys())
    n_algos = len(algo_names)
    p_values = np.zeros((n_algos, n_algos))
    
    for i, algo1 in enumerate(algo_names):
        for j, algo2 in enumerate(algo_names):
            if i != j:
                _, p = stats.ttest_ind(
                    results_dict[algo1][metric],
                    results_dict[algo2][metric]
                )
                p_values[i, j] = p
            else:
                p_values[i, j] = 1.0
    
    return pd.DataFrame(p_values, index=algo_names, columns=algo_names)

print("Pairwise t-tests (Silhouette) — Covertype:")
print(perform_statistical_tests(results_cov).round(4))

print("\nPairwise t-tests (Silhouette) — Fashion-MNIST:")
print(perform_statistical_tests(results_fm).round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

metrics = ['silhouette', 'v_measure']
metric_names = ['Silhouette Score', 'V-Measure']
datasets_info = [
    (results_cov, "Covertype"),
    (results_fm, "Fashion-MNIST")
]

for i, (metric, metric_name) in enumerate(zip(metrics, metric_names)):
    for j, (results_dict, dataset_name) in enumerate(datasets_info):
        ax = axes[i, j]
        for algo_name, df in results_dict.items():
            ax.plot(df['seed'], df[metric], marker='o', label=algo_name, linewidth=1.5)
        ax.set_xlabel('Random Seed')
        ax.set_ylabel(metric_name)
        ax.set_title(f'{dataset_name} — {metric_name} Stability')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/convergence_stability.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def create_detailed_comparison():
    all_results = []
    
    datasets_info = [
        (results_cov, "Covertype"),
        (results_fm, "Fashion-MNIST")
    ]
    
    for results_dict, dataset_name in datasets_info:
        for algo_name, df in results_dict.items():
            all_results.append({
                'Dataset': dataset_name,
                'Algorithm': algo_name,
                'Runtime_mean': df['runtime'].mean(),
                'Runtime_std': df['runtime'].std(),
                'Silhouette_mean': df['silhouette'].mean(),
                'Silhouette_std': df['silhouette'].std(),
                'V_Measure_mean': df['v_measure'].mean(),
                'V_Measure_std': df['v_measure'].std(),
                'ARI_mean': df['ari'].mean(),
                'ARI_std': df['ari'].std()
            })
    
    detailed_df = pd.DataFrame(all_results)
    detailed_df.to_csv('results/detailed_comparison.csv', index=False)
    return detailed_df

detailed = create_detailed_comparison()
print(detailed.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 10))

np.random.seed(42)
idx_viz = np.random.choice(len(X_cov_50k), 5000, replace=False)
X_viz = X_cov_50k[idx_viz]
y_viz = y_cov_50k[idx_viz]
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_viz)

for col, (name, (algo_class, params)) in enumerate(algorithms.items()):
    if algo_class in [MiniBatchKMeans, MiniBatchFuzzyCMeans]:
        model = algo_class(n_clusters=7, random_state=42, **params)
        model.fit(X_viz)
    else:
        model = algo_class(X_viz, n_clusters=7, random_state=42, **params)
        model.fit()
    
    pred = model.predict(X_viz)
    centroids_pca = pca.transform(model.get_centroids())
    
    axes[0, col].scatter(X_pca[:, 0], X_pca[:, 1], c=y_viz, cmap='tab10', alpha=0.3, s=5)
    axes[0, col].set_title(f'True Labels')
    
    axes[1, col].scatter(X_pca[:, 0], X_pca[:, 1], c=pred, cmap='tab10', alpha=0.3, s=5)
    axes[1, col].scatter(centroids_pca[:, 0], centroids_pca[:, 1], 
                         c='red', marker='X', s=200, edgecolors='black', linewidth=2)
    axes[1, col].set_title(f'{name}')

axes[0, 0].set_ylabel('True Labels')
axes[1, 0].set_ylabel('Predicted Clusters')

plt.suptitle('Covertype — Clustering Results (PCA projection)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/clustering_pca_covertype.png', dpi=300, bbox_inches='tight')
plt.show()

# Part 5: Scalability Benchmark

Direct comparison of standard KMeans vs MiniBatch KMeans on a large synthetic dataset (1M points) to clearly show the computational advantage of mini-batches.

In [ ]:
from sklearn.datasets import make_blobs

X_large, y_large = make_blobs(n_samples=1000000, centers=3, n_features=2, random_state=42)

start_time = time.time()
kmeans = KMeansClustering(x_train=X_large, n_clusters=3)
kmeans.fit()
standard_time = time.time() - start_time

start_time = time.time()
mb_kmeans = MiniBatchKMeans(n_clusters=3, max_iter=100, batch_size=1024, random_state=42)
mb_kmeans.fit(X_large)
minibatch_time = time.time() - start_time

print(f"Standard KMeans Time:   {standard_time:.4f} seconds")
print(f"MiniBatch KMeans Time:  {minibatch_time:.4f} seconds")
print(f"Speedup Factor:         {standard_time / minibatch_time:.2f}x")

plt.figure(figsize=(8, 5))
plt.bar(['Standard KMeans', 'MiniBatch KMeans'], [standard_time, minibatch_time], 
        color=['skyblue', 'salmon'])
plt.ylabel('Runtime (seconds)')
plt.title('Execution Time Comparison (N=1,000,000)')
plt.savefig('results/scalability_benchmark.png', dpi=300, bbox_inches='tight')
plt.show()

# Part 6: Image Segmentation (Computer Vision Application)

Applying clustering to image segmentation — each pixel is treated as a data point in RGB color space. The algorithm groups pixels with similar colors, effectively segmenting the image into regions.

In [ ]:
from PIL import Image
import os

def load_and_preprocess_image(image_path, resize_factor=0.5):
    img = Image.open(image_path)
    new_size = (int(img.size[0] * resize_factor), int(img.size[1] * resize_factor))
    img = img.resize(new_size, Image.LANCZOS)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    image_array = np.array(img)
    original_shape = image_array.shape
    pixels = image_array.reshape(-1, 3).astype(np.float64) / 255.0
    print(f"Image loaded: {original_shape[0]}x{original_shape[1]} pixels")
    print(f"Total pixels for clustering: {pixels.shape[0]}")
    return image_array, pixels, original_shape

def load_image_with_spatial_features(image_path, resize_factor=0.5, spatial_weight=0.1):
    img = Image.open(image_path)
    new_size = (int(img.size[0] * resize_factor), int(img.size[1] * resize_factor))
    img = img.resize(new_size, Image.LANCZOS)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    image_array = np.array(img)
    h, w = image_array.shape[:2]
    
    x_coords = np.linspace(0, 1, w)
    y_coords = np.linspace(0, 1, h)
    xx, yy = np.meshgrid(x_coords, y_coords)
    
    rgb_normalized = image_array.reshape(-1, 3).astype(np.float64) / 255.0
    spatial = np.column_stack([xx.ravel(), yy.ravel()]) * spatial_weight
    pixels = np.column_stack([rgb_normalized, spatial])
    
    return image_array, pixels, image_array.shape

def segment_image(img_array, pixels, img_shape, model, n_clusters):
    labels = model.predict(pixels)
    centroids = model.get_centroids()
    segmented = centroids[labels][:, :3]
    segmented = np.clip(segmented, 0, 1)
    segmented_img = (segmented.reshape(img_shape[0], img_shape[1], 3) * 255).astype(np.uint8)
    return segmented_img, labels

def visualize_segmentation(img_array, segmented, labels, method_name, n_clusters):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img_array)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(segmented)
    axes[1].set_title(f'{method_name} ({n_clusters} segments)')
    axes[1].axis('off')
    label_img = labels.reshape(img_array.shape[0], img_array.shape[1])
    axes[2].imshow(label_img, cmap='tab10')
    axes[2].set_title('Cluster Map')
    axes[2].axis('off')
    plt.tight_layout()
    return fig

image_path = 'datasets/image.png'

In [ ]:
if os.path.exists(image_path):
    img_array, pixels, img_shape = load_and_preprocess_image(image_path, resize_factor=0.5)
    n_clusters = 5
    
    # KMeans
    print(f"Running K-Means segmentation with {n_clusters} clusters...")
    start_time = time.time()
    kmeans_img = KMeansClustering(pixels, n_clusters=n_clusters, max_iter=100, random_state=42)
    kmeans_img.fit()
    print(f"K-Means: {time.time() - start_time:.3f}s")
    segmented_kmeans, labels_kmeans = segment_image(img_array, pixels, img_shape, kmeans_img, n_clusters)
    fig = visualize_segmentation(img_array, segmented_kmeans, labels_kmeans, "K-Means", n_clusters)
    plt.savefig('results/kmeans_image_segmentation.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # MiniBatch KMeans
    print(f"Running MiniBatch K-Means segmentation...")
    start_time = time.time()
    mb_kmeans_img = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, max_iter=100, random_state=42)
    mb_kmeans_img.fit(pixels)
    print(f"MiniBatch K-Means: {time.time() - start_time:.3f}s")
    segmented_mb, labels_mb = segment_image(img_array, pixels, img_shape, mb_kmeans_img, n_clusters)
    fig = visualize_segmentation(img_array, segmented_mb, labels_mb, "MiniBatch K-Means", n_clusters)
    plt.savefig('results/minibatch_image_segmentation.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # FCM
    print(f"Running Fuzzy C-Means segmentation...")
    start_time = time.time()
    fcm_img = FuzzyCMeans(pixels, n_clusters=n_clusters, max_iter=100, random_state=42)
    fcm_img.fit()
    print(f"Fuzzy C-Means: {time.time() - start_time:.3f}s")
    segmented_fcm, labels_fcm = segment_image(img_array, pixels, img_shape, fcm_img, n_clusters)
    fig = visualize_segmentation(img_array, segmented_fcm, labels_fcm, "Fuzzy C-Means", n_clusters)
    plt.savefig('results/fcm_image_segmentation.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print(f"Image not found at: {image_path}")

In [ ]:
if os.path.exists(image_path):
    fig, axes = plt.subplots(2, 2, figsize=(16, 16))
    
    axes[0, 0].imshow(img_array)
    axes[0, 0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(segmented_kmeans)
    axes[0, 1].set_title(f'K-Means ({n_clusters} segments)', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')
    
    axes[1, 0].imshow(segmented_mb)
    axes[1, 0].set_title(f'MiniBatch K-Means ({n_clusters} segments)', fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(segmented_fcm)
    axes[1, 1].set_title(f'Fuzzy C-Means ({n_clusters} segments)', fontsize=14, fontweight='bold')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.savefig('results/all_methods_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
if os.path.exists(image_path):
    cluster_numbers = [3, 5, 7, 10]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 16))
    axes = axes.ravel()
    
    for idx, n_clust in enumerate(cluster_numbers):
        print(f"Segmenting with {n_clust} clusters...")
        kmeans_test = KMeansClustering(pixels, n_clusters=n_clust, max_iter=100, random_state=42)
        kmeans_test.fit()
        seg_img, _ = segment_image(img_array, pixels, img_shape, kmeans_test, n_clust)
        axes[idx].imshow(seg_img)
        axes[idx].set_title(f'K-Means — {n_clust} Clusters', fontsize=12, fontweight='bold')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('results/kmeans_multiple_clusters.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
if os.path.exists(image_path):
    img_array_spatial, pixels_spatial, img_shape_spatial = load_image_with_spatial_features(
        image_path, resize_factor=0.5, spatial_weight=0.2
    )
    
    n_clusters = 5
    kmeans_spatial = KMeansClustering(pixels_spatial, n_clusters=n_clusters, max_iter=100, random_state=42)
    kmeans_spatial.fit()
    segmented_spatial, labels_spatial = segment_image(img_array_spatial, pixels_spatial, 
                                                     img_shape_spatial, kmeans_spatial, n_clusters)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    axes[0].imshow(img_array)
    axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    axes[1].imshow(segmented_kmeans)
    axes[1].set_title('K-Means (Color Only)', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    axes[2].imshow(segmented_spatial)
    axes[2].set_title('K-Means (Color + Spatial)', fontsize=14, fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig('results/spatial_vs_color_only.png', dpi=300, bbox_inches='tight')
    plt.show()